# 1. What you'll learn

        - implement a two-layer forward pass in NumPy
- derive and code backpropagation updates
- match the same network in compact PyTorch

        **The one question this notebook answers:** What exactly flows forward and backward through a neural network?

# 2. Setup

This lesson keeps an mlp from numpy to pytorch small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import torch
from torch import nn

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Eight hundred noisy two-moons points create a nonlinear binary task. Two coordinates are inputs and the moon identity is the target.

**Small example:** For x=[2,-1], weights [0.5,0.25] and bias 0.1, one hidden pre-activation is 2×.5-1×.25+.1=.85; ReLU keeps .85 and would pass gradient 1.

In [ ]:
X,y=make_moons(n_samples=800,noise=.22,random_state=SEED);print("Shape:",X.shape,"balance:",np.bincount(y).tolist());print(pd.DataFrame(X,columns=["x1","x2"]).assign(target=y).head().round(3))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",s=14);axes[0].set_title("Nonlinear moons")
axes[1].hist(X[:,0],bins=25,color="#176b66");axes[1].set_title("x1 distribution")
axes[2].hist(X[:,1],bins=25,color="#d38b45");axes[2].set_title("x2 distribution");plt.tight_layout();plt.show()

**Takeaway:** A straight boundary cannot trace both curved classes.

**Takeaway:** Inputs occupy similar scales, but we still fit scaling on training only.

**Takeaway:** Balanced classes make chance accuracy 50%.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
Xa,Xb,ya,yb=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED);scaler=StandardScaler().fit(Xa);Xa=scaler.transform(Xa);Xb=scaler.transform(Xb);print("Train/test:",Xa.shape,Xb.shape)

# 6. Train

        Training turns the assumptions behind an mlp from numpy to pytorch into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Multiply inputs by first-layer weights and apply ReLU.
2. Map hidden activations to a logit and sigmoid probability.
3. Differentiate binary cross-entropy backward through sigmoid and ReLU.
4. Update every weight opposite its gradient and repeat.

In [ ]:
rng=np.random.default_rng(SEED);W1=rng.normal(0,.3,(2,12));b1=np.zeros((1,12));W2=rng.normal(0,.3,(12,1));b2=np.zeros((1,1));Y=ya[:,None];losses=[]
for epoch in range(700):
 H0=Xa@W1+b1;H=np.maximum(H0,0);logit=H@W2+b2;P=1/(1+np.exp(-logit));loss=-np.mean(Y*np.log(P+1e-9)+(1-Y)*np.log(1-P+1e-9));losses.append(loss)
 dlog=(P-Y)/len(Y);dW2=H.T@dlog;db2=dlog.sum(0,keepdims=True);dH=(dlog@W2.T)*(H0>0);dW1=Xa.T@dH;db1=dH.sum(0,keepdims=True);W1-=.08*dW1;b1-=.08*db1;W2-=.08*dW2;b2-=.08*db2
def numpy_prob(V): return 1/(1+np.exp(-(np.maximum(V@W1+b1,0)@W2+b2))).ravel()
print("Loss first/last:",round(losses[0],3),round(losses[-1],3),"test accuracy:",round(accuracy_score(yb,numpy_prob(Xb)>.5),3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
torch.manual_seed(SEED);net=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,1));opt=torch.optim.Adam(net.parameters(),lr=.03);tx=torch.tensor(Xa,dtype=torch.float32);ty=torch.tensor(ya[:,None],dtype=torch.float32)
torch_losses=[]
for _ in range(180):
 opt.zero_grad();loss=nn.functional.binary_cross_entropy_with_logits(net(tx),ty);loss.backward();opt.step();torch_losses.append(loss.item())
with torch.no_grad(): tprob=torch.sigmoid(net(torch.tensor(Xb,dtype=torch.float32))).numpy().ravel()
xx,yy=np.meshgrid(np.linspace(Xb[:,0].min()-.5,Xb[:,0].max()+.5,150),np.linspace(Xb[:,1].min()-.5,Xb[:,1].max()+.5,150));grid=np.c_[xx.ravel(),yy.ravel()]
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(losses,label="NumPy");axes[0].plot(np.linspace(0,699,len(torch_losses)),torch_losses,label="PyTorch");axes[0].set_title("Training loss");axes[0].legend()
axes[1].contourf(xx,yy,numpy_prob(grid).reshape(xx.shape),levels=20,cmap="RdBu",alpha=.7);axes[1].scatter(Xb[:,0],Xb[:,1],c=yb,cmap="coolwarm",s=10);axes[1].set_title("NumPy probability")
axes[2].bar(["chance","NumPy","PyTorch"],[.5,accuracy_score(yb,numpy_prob(Xb)>.5),accuracy_score(yb,tprob>.5)],color=[".6","#176b66","#d38b45"]);axes[2].set_ylim(.45,1);axes[2].set_title("Held-out accuracy");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Bad initialization can kill every ReLU, producing zero gradients and a constant classifier. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
dead_W1=np.full((2,12),-8.0);dead_b1=np.full((1,12),-8.0);dead_H=np.maximum(Xa@dead_W1+dead_b1,0);active=(dead_H>0).mean();constant=np.repeat(ya.mean(),len(yb));print(f"Active ReLUs after hostile initialization={active:.1%}; resulting constant accuracy={accuracy_score(yb,constant>.5):.3f}; healthy={accuracy_score(yb,numpy_prob(Xb)>.5):.3f}")

**Use this when…** nonlinear combinations of features matter and sufficient labelled data support learned representations.

        **Don't use this when…** data are tiny, tabular boosting is stronger and easier, or interpretability and calibrated uncertainty dominate.

        ## Try this

        1. Replace ReLU with tanh and compare learning.
2. Change hidden width from 2 to 64.
3. Check one NumPy gradient with finite differences.